In [1]:
import datetime
import os
import uuid

from dotenv import load_dotenv
import xarray as xr
from xsdba.adjustment import QuantileDeltaMapping

In [3]:
load_dotenv()

#Full parsed ERA5
ERA5_URI = os.environ["POREALLAS_PARSED_ERA5_URI"]
#Recent ERA5 2026
ERA5_SIM_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/26_era5_daily_running.zarr"
#Reference Set: GMFD
GMFD_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/gmfd_parsed.zarr"
OUT_ZARR = "/home/emily_zuetell/projects/poreallas/data/2606to08_era5_running_adj.zarr"
#Clean GMFD Record Years
HISTREF_START_YEAR = 1981
HISTREF_STOP_YEAR = 1997
#Inclusive 20-yr simulation period distribution(ERA5)
SIM_START_YEAR = 2007
SIM_STOP_YEAR = 2026
QDM_N_QUANTILES = 100
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()

In [4]:
gmfd = xr.open_dataset(
    GMFD_URI,
    engine="zarr",
    chunks={},
    )
# Fill extreme values
gmfd = gmfd.sortby("latitude").chunk({"latitude": -1, "longitude": 30, "time": -1})
gmfd = (
    gmfd.where(gmfd["tas"] < 1000)
    .interpolate_na(dim="latitude", method="linear")
    .compute()
)

In [5]:
# Parsed Hist ERA5
era5_hist = xr.open_dataset(
    ERA5_URI,
    engine="zarr",
    chunks={},
    backend_kwargs={"storage_options": {"token": "anon"}},
)

# Running ERA5 (2026)
era5_sim = xr.open_dataset(
    ERA5_SIM_URI,
    engine="zarr",
    chunks={},
)
#Concatenate the ERA5 2026 data with the historical ERA5 data
era5_sim = xr.concat([era5_hist, era5_sim], dim="time")

ref = gmfd.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
hist = era5_hist.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
sim = era5_sim.sel(time=slice(str(SIM_START_YEAR), str(SIM_STOP_YEAR)))

# # "time" dim cannot be chunked for QDM.
ref = ref.chunk({"time": -1})
hist = hist.chunk({"time": -1})
sim = sim.chunk({"time": -1})

qdm = QuantileDeltaMapping.train(
    ref["tas"], hist["tas"], nquantiles=QDM_N_QUANTILES, kind="+", group="time.month"
)



In [6]:
sim_adj = qdm.adjust(sim["t2m"])

sim_adj.name = "tas"
sim_adj = sim_adj.to_dataset()

# Add additional general metadata.
sim_adj.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted ERA5 (running 2026) climate fields",
}
sim_adj["tas"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted ERA5 (running 2026) tas field",
    "poreallas_adjustment_method": "QDM",
    "poreallas_histref_start_year": HISTREF_START_YEAR,
    "poreallas_histref_stop_year": HISTREF_STOP_YEAR,
    "poreallas_sim_start_year": SIM_START_YEAR,
    "poreallas_sim_stop_year": SIM_STOP_YEAR,
    "poreallas_qdm_nquantiles": QDM_N_QUANTILES,
    "poreallas_ref_uri": GMFD_URI,
    "poreallas_hist_uri": ERA5_URI,
    "poreallas_sim_uri": ERA5_SIM_URI,
}

sim_adj = sim_adj.chunk("auto").compute()

OUT_ZARR = "/home/emily_zuetell/projects/poreallas/data/26_era5_running_adj.zarr"
sim_adj.to_zarr(OUT_ZARR, consolidated=True)
print(f"Output written to {OUT_ZARR}")

Output written to /home/emily_zuetell/projects/poreallas/data/26_era5_running_adj.zarr


/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
